# Interactive Metabolic Flux Dashboard

## Overview
This dashboard was developed to explore biomass production across metabolic exchange closed reactions in a breast cancer cell.

The application allows dynamic exploration of metabolic behavior by highlighting individual reactions and comparing their associated biomass production levels.

## Features
- Interactive reaction selection
- Dynamic Plotly visualization
- Biomass production analysis
- Metabolic reaction exploration

## Technologies
- Python
- Dash
- Plotly
- Pandas

## Results: In the interactive dashboard you can see that there are points in all the space, these points represents a metabolite and biomass production when the metabolite is closed. I the lowest levels of the graph I identified aminoacids such as leucine and asparagine that are essential for biomass production, whithout these reactions ther is no growth. This results aligned with the literature and demostrates the capability of Metacolic Models in silico to represent real phenotypes.

In [ ]:
!pip install pandas dash plotly  # Solo instalar Dash y Plotly

import pandas as pd
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.express as px
import numpy as np

# Cargar tu archivo CSV (asegurándote de que el archivo CSV esté en el directorio adecuado)
df = pd.read_csv('/Users/eduardoruiz/Documents/MCBCI/MCBCI2/Sistemas metabólicos/Modelos cancer/exchange_reactions_biomass.csv')

# Extraer las columnas necesarias
etiquetas = df.iloc[:, 1]  # Columna de etiquetas
biomasa = df.iloc[:, 2]   # Columna de producción de biomasa

# Crear un DataFrame con las columnas necesarias
df = pd.DataFrame({
    "Etiquetas": etiquetas,
    "Biomasa": biomasa
})

# Visualizar las primeras filas del archivo para verificar los datos
print(df.head())

# Crear la aplicación Dash
import dash
app = dash.Dash(__name__)

# Diseño de la aplicación
app.layout = html.Div([
    html.H1("Dashboard de Flujos Metabólicos", style={'textAlign': 'center'}),
    dcc.Dropdown(
        id='dropdown-etiqueta',
        options=[{'label': etiqueta, 'value': etiqueta} for etiqueta in df['Etiquetas']],
        value=df['Etiquetas'][0],
        placeholder="Selecciona una reacción...",
        style={'width': '50%', 'margin': '0 auto'}
    ),
    dcc.Graph(id='grafico-biomasa'),
    html.Div(id='info-seleccion', style={'textAlign': 'center', 'marginTop': '20px', 'fontSize': '18px', 'fontWeight': 'bold'})
])

# Callback para actualizar gráfico y texto
@app.callback(
    [Output('grafico-biomasa', 'figure'),
     Output('info-seleccion', 'children')],
    [Input('dropdown-etiqueta', 'value')]
)
def actualizar_grafico(etiqueta_seleccionada):
    # Filtrar la reacción seleccionada
    df_seleccionado = df[df['Etiquetas'] == etiqueta_seleccionada]
    
    # Crear un gráfico de dispersión
    fig = px.scatter(
        df,
        x="Etiquetas",
        y="Biomasa",
        title="Producción de biomasa para todas las reacciones",
        labels={"Etiquetas": "Reacción", "Biomasa": "Producción de biomasa"},
        template="plotly"
    )
    
    # Resaltar el punto seleccionado
    fig.add_scatter(
        x=df_seleccionado['Etiquetas'],
        y=df_seleccionado['Biomasa'],
        mode='markers',
        marker=dict(size=12, color='red'),
        name=f'Selección: {etiqueta_seleccionada}'
    )
    
    # Configurar el tamaño del gráfico, eliminar las etiquetas del eje X y agregar números
    fig.update_layout(
    height=600,  # Aumentar la altura del gráfico
    width=1000,  # Aumentar el ancho del gráfico
    xaxis=dict(
        showticklabels=True,  # Mostrar los números en el eje X
        tickmode='array',  # Usar una lista personalizada para los ticks
        tickvals=np.arange(0, len(df), 50),  # Agregar números como marcas
        ticktext=[str(i) for i in np.arange(0, len(df), 50)]  # Usar esos números como etiquetas
        )
    )
    
    # Información adicional
    info = f"La biomasa producida por {etiqueta_seleccionada} es {df_seleccionado['Biomasa'].values[0]:.2f}."
    return fig, info

# Ejecutar la aplicación en modo Jupyter
app.run_server(debug=True)




[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
         Etiquetas     Biomasa
0   benzo[a]pyrene  701.269597
1      naphthalene  701.269597
2     aflatoxin B1  701.269597
3  trichloroethene  701.269597
4     bromobenzene  701.269597
